In [ ]:
#크롤링 가능한 사이트 확인 : 사이트 주소/robot.txt

### 크롤링 예제
1. requests라이브러리를 이용해서 `moons-86.iptime.org:8080` 요청을 보낸다.
2. 응답 받은 데이터를 BeautifulSoup을 이용하여 데이터를 파싱
3. id가 product_1001인 태그를 찾아서 h2, p의 모든 콘탠츠 데이터를 추출한다.
4. 모든 상품의 정보를 추출
    - 데이터프레임으로 생성
    - csv 파일로 저장

In [27]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

In [ ]:
#1.서버에 요청을 보낸다.
res=requests.get('http://moons-86.iptime.org:8080/')

In [29]:
html_text=res.text

In [73]:
#2. BeautifulSoup을 이용해서 데이터를 파싱
soup=bs(html_text,'html.parser' )
type(soup)

bs4.BeautifulSoup

In [75]:
#id가 product-1001인 태그를 찾아라 -> div 태그에서 id가 product-1001
div_data=soup.find(
    'div',
    attrs={
        'id':'product-1001'
    }
)

In [81]:
div_data.get_text().split('\n')

['',
 '스마트 워치 X100',
 '카테고리: 전자기기',
 '가격: 150,000원',
 '평점: ★★★★☆ (4.5/5)',
 '상세 보기',
 '']

In [83]:
#h2, p 태그를 모두 찾아서 각각 텍스트를 추출
p_list=div_data.find_all(
    ['h2','p']
)

In [ ]:
#p_list에서 각각의 원소들의 콘텐츠 데이터를 추출
#1)
[p.get_text() for p in p_list]

['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)']

In [ ]:
#2)
list(
    map(
        lambda x : x.get_text(),
        p_list
    )
)

['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)']

In [88]:
#사이트의 모든 상품의 정보를 가져온다.
#접근방법 1 : div 태그 중 class가 product-item인 태그를 모두 찾는다.
div_list=soup.find_all(
    'div',
    attrs={
        'class':"product-item"
    }
)

In [89]:
#div_list는 각각의 원소가 아이템의 정보를 가지고 있다.
# div_list[0]     #id가 product-1001과 같은 태그 -> 위의 작업과 동일한 작업을 반복 실행하여 빈 리스트애 결과를 추가

values=[]

for div in div_list:
    #h2, p 태그를 모두 찾는다.
    p_list = div.find_all(
        ['h2','p']
    )
    #p_list에서 각각의 원소들에서 텍스트를 추출
    value=[p.get_text() for p in p_list]
    #values에 value를 추가
    values.append(value)

values

[['스마트 워치 X100', '카테고리: 전자기기', '가격: 150,000원', '평점: ★★★★☆ (4.5/5)'],
 ['무선 이어폰 Pro', '카테고리: 전자기기', '가격: 89,000원', '평점: ★★★★★ (5.0/5)'],
 ['고급 가죽 지갑', '카테고리: 패션 잡화', '가격: 75,000원', '평점: ★★★★☆ (4.0/5)'],
 ['에코백 디자인 컬렉션', '카테고리: 패션 잡화', '가격: 25,000원', '평점: ★★★★☆ (4.2/5)'],
 ['유기농 커피 원두 500g', '카테고리: 식품', '가격: 18,000원', '평점: ★★★★★ (4.8/5)']]

In [94]:
df=pd.DataFrame(
    values,
    columns=['상품명','카테고리','가격','평점']
)

In [ ]:
#Series에서 replace() 함수의 기준운? 각각의 value
#문자를 기준으로 replace를 사용하려면? 1. 반복문 이용, 2. map()함수 이용, 3. .str을 이용하여 문자열 함수에 접근
for i in df['카테고리']:
    print(i.replace('카테고리: ',''))
    

# [i.replace('카테고리: ','') for i in df['카테고리']]

전자기기
전자기기
패션 잡화
패션 잡화
식품


In [100]:
#python에 내장된 map() 함수는 데이터를 가지고 있지 않기 때문에 인자에 데이터를 입력
#Series에 내장된 map()함수는 Series안에 데이터가 이미 존재하기 때문에 인자에는 함수만 입력
df['가격'].map(
    lambda x : x.replace('가격: ','')
)

0    150,000원
1     89,000원
2     75,000원
3     25,000원
4     18,000원
Name: 가격, dtype: str

In [101]:
df['평점'].str.replace('평점: ','')

0    ★★★★☆ (4.5/5)
1    ★★★★★ (5.0/5)
2    ★★★★☆ (4.0/5)
3    ★★★★☆ (4.2/5)
4    ★★★★★ (4.8/5)
Name: 평점, dtype: str

In [107]:
#데이터프레임에서 map() 함수를 이용하면 value들이 각각 입력

df.map(
    lambda x : x.split(':')[-1].lstrip()
)

,상품명,카테고리,가격,평점
0,스마트 워치 X100,전자기기,"150,000원",★★★★☆ (4.5/5)
1,무선 이어폰 Pro,전자기기,"89,000원",★★★★★ (5.0/5)
2,고급 가죽 지갑,패션 잡화,"75,000원",★★★★☆ (4.0/5)
3,에코백 디자인 컬렉션,패션 잡화,"25,000원",★★★★☆ (4.2/5)
4,유기농 커피 원두 500g,식품,"18,000원",★★★★★ (4.8/5)


In [109]:
#접근 방식2 -> 상품명, 가격, 카테고리, 평점, 하이퍼링크를 각각의 class의 값들로 접근하여 데이터프레임 생성

#product_list class를 가진 div를 추출(모든 상품의 정보가 담겨있는 div영역을 선택)

div_data=soup.find(
    'div',
    attrs={
        'class':'product-list'
    }
)

In [112]:
#상품명에 접근 : class의 값이 product_title인 태그에 접근 (모두 찾는다.)
title_list=div_data.find_all(
    attrs={
        'class':'product-title'
    }
)

In [114]:
[title.get_text() for title in title_list]

['스마트 워치 X100', '무선 이어폰 Pro', '고급 가죽 지갑', '에코백 디자인 컬렉션', '유기농 커피 원두 500g']

In [127]:
#a태그에서 href라는 속성의 값을 추출하려면? Tag['속성명']
link_list=div_data.find_all(
    attrs={
        'class':'product-link'
    }
)

In [128]:
links=[]
for link in link_list:
    # print('https://moons-86.iptime.org:8080' + link['href'])
    links.append('https://moons-86.iptime.org:8080' + link['href'])
links

['https://moons-86.iptime.org:8080/products/1001',
 'https://moons-86.iptime.org:8080/products/1002',
 'https://moons-86.iptime.org:8080/products/1003',
 'https://moons-86.iptime.org:8080/products/1004',
 'https://moons-86.iptime.org:8080/products/1005']

In [133]:
class_list = [ 'product-title', 'product-category', 'product-price', 'product-rating' , 'product-link']
base_url = "http://moons-86.iptime.org"
dict_data = {}
# 반복 실행하면서 빈 딕셔너리에 key : [] 값들을 추가 
for cls in class_list:
    _list = div_data.find_all(
        attrs= {
            'class' : cls
        }
    )
    # cls가 product-link 라면 href 속성의 값들을 리스트로 생성
    if cls == 'product-link':
        value = [ base_url + link['href']  for link in _list ]
    else:
        value = [ data.get_text().split(':')[-1].lstrip() for data in _list ]
    
    # key 값은 cls에서 -로 잘라주고 뒤의 값들을 사용
    key_data = cls.split('-')[-1]

    dict_data[key_data] = value

dict_data

{'title': ['스마트 워치 X100',
  '무선 이어폰 Pro',
  '고급 가죽 지갑',
  '에코백 디자인 컬렉션',
  '유기농 커피 원두 500g'],
 'category': ['전자기기', '전자기기', '패션 잡화', '패션 잡화', '식품'],
 'price': ['150,000원', '89,000원', '75,000원', '25,000원', '18,000원'],
 'rating': ['★★★★☆ (4.5/5)',
  '★★★★★ (5.0/5)',
  '★★★★☆ (4.0/5)',
  '★★★★☆ (4.2/5)',
  '★★★★★ (4.8/5)'],
 'link': ['http://moons-86.iptime.org/products/1001',
  'http://moons-86.iptime.org/products/1002',
  'http://moons-86.iptime.org/products/1003',
  'http://moons-86.iptime.org/products/1004',
  'http://moons-86.iptime.org/products/1005']}

In [134]:
pd.DataFrame(dict_data)

,title,category,price,rating,link
0,스마트 워치 X100,전자기기,"150,000원",★★★★☆ (4.5/5),http://moons-86.iptime.org/products/1001
1,무선 이어폰 Pro,전자기기,"89,000원",★★★★★ (5.0/5),http://moons-86.iptime.org/products/1002
2,고급 가죽 지갑,패션 잡화,"75,000원",★★★★☆ (4.0/5),http://moons-86.iptime.org/products/1003
3,에코백 디자인 컬렉션,패션 잡화,"25,000원",★★★★☆ (4.2/5),http://moons-86.iptime.org/products/1004
4,유기농 커피 원두 500g,식품,"18,000원",★★★★★ (4.8/5),http://moons-86.iptime.org/products/1005


In [135]:
#csv파일로 저장시 인덱스가 포함되어 저장 -> 의마가 없는 인덱스일수도 있지만 의미가 존재하는 경우
#index매개변수 존재 -> 인덱스를 저장할것인가?
df.to_csv('test.csv',index=False)

In [138]:
#csv파일에 인덱스(이름 없는 인덱스) 부분이 저장되어있는 경우 인덱스 부분도 컬럼을 인식해서 로드
#index_col 매개변수 : 인덱스로 사용할 컬럼은 선택(위치/컬럼명) 다중선택 가능
#usecols 매개변수 : 데이터프레임에서 컬럼의 필터(위치/컬럼명)
pd.read_csv('test.csv')

,상품명,카테고리,가격,평점
0,스마트 워치 X100,카테고리: 전자기기,"가격: 150,000원",평점: ★★★★☆ (4.5/5)
1,무선 이어폰 Pro,카테고리: 전자기기,"가격: 89,000원",평점: ★★★★★ (5.0/5)
2,고급 가죽 지갑,카테고리: 패션 잡화,"가격: 75,000원",평점: ★★★★☆ (4.0/5)
3,에코백 디자인 컬렉션,카테고리: 패션 잡화,"가격: 25,000원",평점: ★★★★☆ (4.2/5)
4,유기농 커피 원두 500g,카테고리: 식품,"가격: 18,000원",평점: ★★★★★ (4.8/5)
